In [1]:
import requests
import time
import sys
import os
import json
import uuid
from psycopg import OperationalError, DatabaseError

from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
import random
from dotenv import load_dotenv
import psycopg
import socket

In [2]:
def is_elasticsearch_ready():
    try:
        socket.getaddrinfo("elasticsearch", None)
        host = "elasticsearch"
    except socket.gaierror:
        # Fallback to local machine if Docker network host isn't found
        host = "localhost"

    try:
        url = f'http://{host}:9200'
        print(url)

        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Elasticsearch service: {e}")
        return False

In [3]:
is_elasticsearch_ready()

http://localhost:9200


True

In [4]:
def is_grafana_ready():
    try:
        socket.getaddrinfo("grafana", None)
        host = "grafana"
    except socket.gaierror:
        # Fallback to local machine if Docker network host isn't found
        host = "localhost"
    try:
        url = f'http://{host}:3000'
        print(url)

        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Grafana service: {e}")
        return False

In [5]:
is_grafana_ready()

http://localhost:3000


True

In [6]:
def is_mage_ready():
    try:
        socket.getaddrinfo("grafana", None)
        host = "grafana"
    except socket.gaierror:
        # Fallback to local machine if Docker network host isn't found
        host = "localhost"

    try:
        url = f'http://{host}:6789'
        print(url)
        
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Mage service: {e}")
        return False

In [7]:
is_mage_ready()

http://localhost:6789


True

In [29]:
def wait_for_services(max_retries=12):  # 12 * 5 seconds = 1 minute total wait time
    retries = 0
    while retries < max_retries:
        if is_elasticsearch_ready() and is_mage_ready() and is_grafana_ready():
            print("All  Elasticsearch and Mage and Grafana are ready!")
            return True
        else:
            print(f"Attempt {retries + 1}/{max_retries}: Services not ready. Waiting 5 seconds...")
            time.sleep(5)
            retries += 1
    print("Max retries reached. Services are not ready.")
    return False

In [30]:
wait_for_services()

http://localhost:9200
http://localhost:6789
http://localhost:3000
All  Elasticsearch and Mage and Grafana are ready!


True

In [ ]:
def run_pipeline_populate_elasticsearch():
    """
    chunking, lammetizing, embedding and indexing data into elastic search via mage pipeline
    """
    url = "http://127.0.0.1:6789/api/pipeline_schedules/1/pipeline_runs/60ac297fd34a457991914d00c79c6a42"
    
    headers = {
        "Content-Type": "application/json"
    }

    print('!----> populate_elasticsearch for magic started', flush=True)
    
    try:
        response = requests.post(url, headers=headers)
        response.raise_for_status()
        print(f'!----> populate_elasticsearch magic finished with code: {response.status_code}', flush=True)
    except Exception as err:
        print(f"An unexpected error occurred magic: {err}", flush=True)
        print("Error details magic:", sys.exc_info(), flush=True)
    finally:
        print("!----> Script execution completed magic.", flush=True)

In [34]:
run_pipeline_populate_elasticsearch()

!----> populate_elasticsearch for magic started
!----> populate_elasticsearch magic finished with code: 200
!----> Script execution completed magic.


In [17]:
import socket

def get_db_connection():
    host = os.getenv("POSTGRES_HOST")

    if not host:
        try:
            socket.getaddrinfo("postgres", None)
            host = "postgres"
        except socket.gaierror:
            # Fallback to local machine if Docker network host isn't found
            host = "localhost"

    try:
        return psycopg.connect(
            host=host,
            dbname=os.getenv("POSTGRES_DB", "ecommerce_chatbot"),
            user=os.getenv("POSTGRES_USER", "user"),
            password=os.getenv("POSTGRES_PASSWORD", "password"),
            connect_timeout=5
        )
    except OperationalError as e:
        print(f"Error: Could not connect to the PostgreSQL database.\nDetails: {e}")
        return None

In [18]:
get_db_connection()

<psycopg.Connection [IDLE] (host=localhost user=user database=ecommerce_chatbot) at 0x779bc6761b20>

In [35]:
def init_db(drop=False):
    conn = get_db_connection()
    try:
        with conn.cursor() as cur:
            if drop:
                cur.execute("DROP TABLE IF EXISTS feedback")
                cur.execute("DROP TABLE IF EXISTS conversations")
                

            cur.execute("""
                CREATE TABLE conversations (
                    id SERIAL PRIMARY KEY,
                    question TEXT NOT NULL,
                    answer TEXT NOT NULL,
                    model TEXT NOT NULL,
                    instructions TEXT NOT NULL,
                    prompt TEXT NOT NULL,
                    prompt_tokens INTEGER NOT NULL,
                    completion_tokens INTEGER NOT NULL,
                    total_tokens INTEGER NOT NULL,
                    response_time FLOAT NOT NULL,
                    cost FLOAT NOT NULL,
                    timestamp TIMESTAMP WITH TIME ZONE NOT NULL
                )
            """)

            cur.execute("""
                CREATE TABLE feedback (
                    id SERIAL PRIMARY KEY,
                    conversation_id INTEGER REFERENCES conversations(id),
                    source TEXT NOT NULL,
                    relevance TEXT,
                    explanation TEXT,
                    score INTEGER,
                    timestamp TIMESTAMP WITH TIME ZONE NOT NULL
                )
            """)

        conn.commit()
        print("Database initialization completed successfully.")

        # Verify table creation by querying information_schema
        with conn.cursor() as cur:
            cur.execute("""
                SELECT table_name FROM information_schema.tables 
                WHERE table_schema = 'public'
            """)
            tables = cur.fetchall()
            print("Tables in the database:", tables)
    except DatabaseError as e:
        print(f"Database error: {e}")
        conn.rollback()  # Rollback in case of error

    finally:
        conn.close()
        print("Database connection closed.")

In [36]:
init_db(drop=True)


Database initialization completed successfully.
Tables in the database: [('conversations',), ('feedback',)]
Database connection closed.


In [40]:
from datetime import datetime
DB_TIMEZONE = datetime.now().astimezone().tzinfo

def save_conversation(record, question):
    timestamp = datetime.now(DB_TIMEZONE)

    conn = get_db_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO conversations (
                    question, answer, model, instructions, prompt,
                    prompt_tokens, completion_tokens, total_tokens,
                    response_time, cost, timestamp
                ) VALUES (
                    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
                )
                RETURNING id
                """,
                (
                    question,
                    record.answer,
                    record.model,
                    record.instructions,
                    record.prompt,
                    record.prompt_tokens,
                    record.completion_tokens,
                    record.total_tokens,
                    record.response_time,
                    record.cost,
                    timestamp,
                ),
            )
            conversation_id = cur.fetchone()[0]
        conn.commit()
    except DatabaseError as e:
            print(f"Database error: {e}")
            conn.rollback()  # Rollback in case of error
    finally:
        conn.close()
    return conversation_id

In [41]:
def save_feedback(conversation_id, source, relevance=None,
                  explanation=None, score=None):
    timestamp = datetime.now(DB_TIMEZONE)

    conn = get_db_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO feedback (
                    conversation_id, source, relevance,
                    explanation, score, timestamp
                ) VALUES (
                    %s, %s, %s, %s, %s, %s
                )
                """,
                (conversation_id, source, relevance,
                 explanation, score, timestamp),
            )
        conn.commit()
    finally:
        conn.close()